# Stain Normalization (Reinhard / Macenko)

Normalizes histology image tiles using the TIAToolbox stain normalization module. Update `BASE_PATH` and `TARGET_IMAGE_PATH` before running.

In [ ]:
%%bash
apt-get -y install libopenjp2-7-dev libopenjp2-tools openslide-tools libpixman-1-dev | tail -n 1
pip install git+https://github.com/TissueImageAnalytics/tiatoolbox.git@develop | tail -n 1
echo "Installation is done."

In [ ]:
"""Import modules required to run the Jupyter notebook."""

from __future__ import annotations

# Clear logger to use tiatoolbox.logger
import logging

if logging.getLogger().hasHandlers():
    logging.getLogger().handlers.clear()

from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import requests
import skimage.color

from tiatoolbox import data, logger
from tiatoolbox.tools import stainnorm
from tiatoolbox.wsicore import wsireader
from tiatoolbox import data, logger


mpl.rcParams["figure.dpi"] = 150  # for high resolution figure in notebook

In [ ]:
"""Import modules required to run the Jupyter notebook."""

from __future__ import annotations

# Clear logger to use tiatoolbox.logger
import logging

if logging.getLogger().hasHandlers():
    logging.getLogger().handlers.clear()

from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import requests
import skimage.color

from tiatoolbox import data, logger
from tiatoolbox.tools import stainnorm
from tiatoolbox.wsicore import wsireader
from tiatoolbox import data, logger


mpl.rcParams["figure.dpi"] = 150  # for high resolution figure in notebook

#######################################################################################
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
import stainnorm

# Define the path to the dataset
base_path = '<BASE_PATH>'
folders = ['Train', 'Test', 'Val']
folds = [f'fold_{i}' for i in range(1, 6)]
classes = ['Glioma', 'Normal']

# Target image for normalization
target_image_path = '<TARGET_IMAGE_PATH>'
target_image = Image.open(target_image_path)
target_image = np.array(target_image)

# Normalization method
method_name = "Reinhard"
stain_normalizer = stainnorm.get_normalizer(method_name)
stain_normalizer.fit(target_image)

# Iterate through all the folders, folds, and classes
for folder in folders:
    for fold in folds:
        for cls in classes:
            class_path = os.path.join(base_path, folder, fold, cls)
            for image_name in os.listdir(class_path):
                image_path = os.path.join(class_path, image_name)
                
                if image_path.endswith('.png'):  # Ensure it's an image file
                    # Open the image using PIL
                    sample = Image.open(image_path)
                    sample = np.array(sample)

                    # Normalize the image
                    normed_sample = stain_normalizer.transform(sample.copy())

                    # Convert the normalized image back to PIL format
                    normed_sample_pil = Image.fromarray(normed_sample)

                    # Save the normalized image back to the same location or a new location
                    normed_image_path = os.path.join(class_path, 'normalized', image_name)
                    os.makedirs(os.path.dirname(normed_image_path), exist_ok=True)
                    normed_sample_pil.save(normed_image_path)

                    # print(f'Normalized and saved image: {normed_image_path}')

print('Normalization complete for the entire dataset.')
